# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [4]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [6]:
feature_columns = contract["features"]

# TODO: crea un preprocesador para árboles con imputación por mediana
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

# TODO: crea otro con imputación y StandardScaler
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_columns)
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

>Porque ajustar el StandardScaler antes de la validación cruzada produce data leakage.

## 3. Modelos candidatos

In [8]:
# TODO: crea pipelines para DecisionTree, LogisticRegression y KNN
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", KNeighborsClassifier())
    ])
}

## 4. Comparación inicial con validación cruzada

In [10]:
# TODO: usa StratifiedKFold de 5 folds y cross_validate
# Reporta accuracy promedio de train, accuracy promedio de validation y desviación estándar
# Guarda el resumen en initial_comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )

    results.append({
        "model": name,
        "mean_train_accuracy": scores["train_score"].mean(),
        "mean_val_accuracy": scores["test_score"].mean(),
        "std_val_accuracy": scores["test_score"].std()
    })

initial_comparison = pd.DataFrame(results)

print(initial_comparison)

                model  mean_train_accuracy  mean_val_accuracy  \
0        DecisionTree               1.0000           0.902217   
1  LogisticRegression               1.0000           0.979064   
2                 KNN               0.9771           0.950739   

   std_val_accuracy  
0          0.058617  
1          0.027713  
2          0.035274  


## 5. Optimización de hiperparámetros

In [12]:
# TODO: define espacios pequeños de búsqueda para los tres modelos
# TODO: ejecuta GridSearchCV con scoring="accuracy"
# Guarda las búsquedas en searches y el resumen en tuned_comparison
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [3, 5, 10],
        "classifier__min_samples_split": [2, 5, 10]
    },
    "LogisticRegression": {
        "classifier__C": [0.1, 1.0, 10.0]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"]
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

searches = {}
tuned_results = []

for name, model in models.items():
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    searches[name] = grid

    best_idx = grid.best_index_
    tuned_results.append({
        "model": name,
        "best_params": grid.best_params_,
        "mean_train_accuracy": grid.cv_results_["mean_train_score"][best_idx],
        "mean_val_accuracy": grid.best_score_,
        "std_val_accuracy": grid.cv_results_["std_test_score"][best_idx]
    })

tuned_comparison = pd.DataFrame(tuned_results)

print(tuned_comparison)

                model                                        best_params  \
0        DecisionTree  {'classifier__max_depth': 5, 'classifier__min_...   
1  LogisticRegression                             {'classifier__C': 1.0}   
2                 KNN  {'classifier__n_neighbors': 7, 'classifier__we...   

   mean_train_accuracy  mean_val_accuracy  std_val_accuracy  
0             1.000000           0.902217          0.058617  
1             1.000000           0.979064          0.027713  
2             0.964773           0.957882          0.040520  


## 6. Selección y persistencia del campeón

In [13]:
# TODO: selecciona el mayor best_score_ de CV
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion_grid = searches[champion_name]

# Obtener el pipeline completo configurado con los mejores hiperparámetros
champion = champion_grid.best_estimator_

# Entrenar el modelo campeón con todo el conjunto de entrenamiento
champion.fit(X_train, y_train)

# TODO: guarda champion_model.joblib, training_metadata.json y cv_results.csv
# 1. Guardar el modelo campeón
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# 2. Guardar los metadatos de entrenamiento
metadata = {
    "champion_model_name": champion_name,
    "best_params": champion_grid.best_params_,
    "best_validation_accuracy": champion_grid.best_score_,
    "features": contract["features"],
    "target": contract["target"]
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=4), encoding="utf-8")

# 3. Guardar el resumen de la comparación para el reporte
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print(f"¡Campeón seleccionado: {champion_name}!")

¡Campeón seleccionado: LogisticRegression!


In [14]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

Se seleccionó Logistic Regression como el modelo campeón debido a que obtuvo el mayor desempeño en la validación cruzada ($\text{accuracy} \approx 0.979$) con la menor variabilidad ($\text{std} \approx 0.027$). Además, demostró no sufrir de sobreajuste crítico, manteniendo un equilibrio excelente entre precisión y generalización. Por último, es un modelo de baja complejidad computacional y de alta interpretabilidad en comparación con las demás alternativas.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.